# Veil In-House Image Detector — Kaggle Training Kernel

Runs the end-to-end pipeline (`detector-trainer/run_pipeline.py`) on Kaggle GPU:
**manifest -> train_resnet -> train_clip -> evaluate**, writing all outputs to
`/kaggle/working` for retrieval via `kaggle kernels output`.

## How the code gets onto Kaggle
This kernel needs the `detector-trainer/` source tree importable. Two supported ways:

1. **Git clone (default below, needs `enable_internet: true`)** — clone the repo
   at run time. Simplest; the `CODE` cell does this.
2. **Attached utility dataset** — push `detector-trainer/` as a Kaggle Dataset
   (`kaggle datasets create/version`), attach it, and point `CODE_DIR` at
   `/kaggle/input/<your-code-dataset>/detector-trainer`. Then set `USE_GIT = False`.

See `detector-trainer/kernels/README.md` for the exact CLI loop.

## 0. Config
Edit `REPO_URL` / `BRANCH` (git path) or `CODE_DIR` (attached-dataset path).

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda)
print('device_count', torch.cuda.device_count())
if torch.cuda.is_available():
    print('name', torch.cuda.get_device_name(0))
    print('capability', torch.cuda.get_device_capability(0))
    print('torch arch_list', torch.cuda.get_arch_list())
    try:
        print('cuda op OK ->', (torch.randn(8, device='cuda')*2).sum().item())
    except Exception as e:
        print('cuda op FAILED ->', repr(e))


In [ ]:
import os, subprocess, sys, glob, json

# Diagnostic: show the mounted input tree (top levels)
for root, dirs, files in sorted(os.walk('/kaggle/input')):
    if root.count('/') - 2 <= 2:
        print(root, '-> dirs:', sorted(dirs)[:20], '| files:', sorted(files)[:12])

# Auto-locate the code wherever Kaggle extracted it
hits = glob.glob('/kaggle/input/**/run_pipeline.py', recursive=True)
assert hits, 'run_pipeline.py not found under /kaggle/input'
CODE_DIR = os.path.dirname(hits[0])
print('CODE_DIR =', CODE_DIR)

OUT_DIR = '/kaggle/working/veil_run'
os.makedirs(OUT_DIR, exist_ok=True)

DIAGNOSE = False  # print GPU/torch info then skip the pipeline


## 1. Get the code

In [ ]:
assert os.path.isdir(CODE_DIR), f'code dir not found: {CODE_DIR}'
print('code dir contents:', sorted(os.listdir(CODE_DIR)))


## 2. Install dependencies
Kaggle has torch preinstalled; this pulls open_clip / imagehash / matching pins.

In [ ]:
# Kaggle stock torch 2.10 dropped sm_60 (P100). Install a torch that supports
# BOTH P100 (sm_60) and T4 (sm_75); install open_clip --no-deps so it can't
# pull torch 2.10 back. The pipeline runs as a subprocess -> picks up new torch.
subprocess.run([sys.executable,'-m','pip','install','-q',
  'torch==2.5.1','torchvision==0.20.1',
  '--index-url','https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','open_clip_torch'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ftfy','regex','imagehash'], check=True)
print('deps installed')
# Pre-flight GPU sanity check in a FRESH process (uses the new torch)
chk = subprocess.run([sys.executable,'-c',
  "import torch;print('torch',torch.__version__);"
  "print('arch',torch.cuda.get_arch_list());"
  "print('op',(torch.randn(8,device='cuda')*2).sum().item())"],
  capture_output=True, text=True)
print(chk.stdout); print(chk.stderr)
assert 'op' in chk.stdout, 'GPU sanity check failed: ' + chk.stderr[-500:]


## 3. Locate the attached datasets
Kaggle mounts `dataset_sources` (see `kernel-metadata.json`) read-only under
`/kaggle/input/<dataset-slug>/`. We pass BOTH roots to `--data-root`; the
pipeline auto-discovers per-generator / category subfolders and classifies them
(real vs each generator). Wild generators (midjourney/dalle3/flux) are routed to
`test_wild` by the split logic. Drop any self-generated dalle3/flux images into a
folder named `dalle3`/`flux` under a data root and they are picked up automatically.

In [ ]:
import glob
# Auto-locate each dataset wherever Kaggle mounts it (/kaggle/input/... or .../datasets/<user>/...)
def find_ds(slug):
    hits = [h for h in glob.glob(f'/kaggle/input/**/{slug}', recursive=True) if os.path.isdir(h)]
    return hits[0] if hits else None
DATA_ROOTS = [p for p in [find_ds('unbiased-tiny-genimage'),
                          find_ds('ai-vs-real-images-dataset'),
                          # Modern-generator fakes (flux/dalle3): folded INTO training
                          # via --wild-generators below so the model learns them.
                          find_ds('veil-detector-wild-fakes')] if p]
assert DATA_ROOTS, 'no training datasets found under /kaggle/input'
for r in DATA_ROOTS:
    print(r, '->', sorted(os.listdir(r))[:20])

# Diverse real-world photos (separate dataset). Each real_* subfolder is passed
# as its OWN --data-root so discover_sources sets source == folder name, which is
# what --wild-real-sources matches. real_*_heldout folders (the DIV2K camera-native
# phone-capture proxy) are held out into test_wild to measure real-photo FPR.
REALS_ROOT = find_ds('veil-detector-reals')
REAL_ROOTS = sorted(glob.glob(os.path.join(REALS_ROOT, 'real_*'))) if REALS_ROOT else []
assert REAL_ROOTS, 'veil-detector-reals not mounted; check kernel dataset_sources'
WILD_REAL_SOURCES = [os.path.basename(p) for p in REAL_ROOTS if p.endswith('_heldout')]
for r in REAL_ROOTS:
    print(r, '->', len(os.listdir(r)), 'images')
print('WILD_REAL_SOURCES =', WILD_REAL_SOURCES)


## 4. Run the full pipeline
All four stages to `/kaggle/working/veil_run`. Adjust epochs for the GPU budget.

In [ ]:
STAGES = 'all'
if DIAGNOSE:
    print('DIAGNOSE mode: skipping pipeline run')
else:  # validation run: discovery+split only (no GPU). Switch to 'all' for full training.
    cmd = [
        sys.executable, 'run_pipeline.py',
        '--data-root', *DATA_ROOTS, *REAL_ROOTS,
        '--wild-real-sources', *WILD_REAL_SOURCES,
        # Hold out ONLY midjourney as the unseen cross-generator test; flux/dalle3
        # are folded into training to close the modern-generator gap.
        '--wild-generators', 'midjourney',
        '--manifest', os.path.join(OUT_DIR, 'manifest.csv'),
        '--out', OUT_DIR,
        '--stages', STAGES,
        '--pretrained',              # ImageNet init for the ResNet baseline
        '--resnet-epochs', '10',
        '--clip-backbone', 'ViT-L-14',
        '--clip-epochs', '200',
        '--num-workers', '2',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=CODE_DIR, check=True)

## 5. Inspect outputs
`report.md`, plots, per-model predictions, checkpoints and `results.json` all land
under `/kaggle/working/veil_run`. Pull them back locally with:
```
kaggle kernels output <KAGGLE_USERNAME>/veil-detector-train -p ./kaggle_out
```

In [ ]:
for p in sorted(glob.glob(os.path.join(OUT_DIR, '**', '*'), recursive=True)):
    if os.path.isfile(p):
        print(p)
print('\n----- report.md -----')
rp = os.path.join(OUT_DIR, 'report', 'report.md')
if os.path.exists(rp):
    print(open(rp).read())